# Processamento em Lote (Batch) - Sinais Neurofisiológicos (fNIRS) | Tarefa: Decision Making
Processamento em lote da resposta hemodinâmica cortical para a amostra completa do estudo (n=54).
A pipeline metodológica inclui:
1. Controlo de qualidade pelo Scalp Coupling Index (SCI > 0.7).
2. Correção de artefactos via TDDR e (quando viável) Regressão de Canais Curtos (SSR).
3. Conversão mBLL e filtragem temporal IIR (0.01 - 0.10 Hz).
4. Isolamento cortical (canais > 1.5 cm).
5. Modelo Linear Geral (GLM) usando a resposta HRF de Glover.
No final, os *Betas* estatísticos (effect) e *p-values* de Oxiemoglobina (HbO) de cada canal são exportados para um dataset consolidado, pronto a cruzar com a fisiologia.

In [3]:
import os
import mne
import mne_nirs
import pandas as pd
import numpy as np
from mne.preprocessing.nirs import optical_density, beer_lambert_law, scalp_coupling_index, temporal_derivative_distribution_repair
from mne_nirs.experimental_design import make_first_level_design_matrix
from mne_nirs.statistics import run_glm
import warnings

# Ignorar os avisos para manter o output do terminal limpo durante o batch
warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

# ==========================================
# 1. Configuração do Lote e Parâmetros Base
# ==========================================
BASE_DIR = r"C:\Users\guilh\Desktop\OneDrive - Universidade de Coimbra\PhD_Project_ARDITI_Guilherme Saldanha\Study2_Data Collection Files_PhD_Guilherme Saldanha_UC-UMa-ITI"
NUM_SUJEITOS = 54  # Processamento da amostra completa
TAREFA = "tk-dm"

LIMIAR_SCI = 0.7
PPF = 0.1
LOW_CUTOFF_HZ = 0.01
HIGH_CUTOFF_HZ = 0.10
DISTANCIA_MIN_LONGA_M = 0.015

EVENTOS_DM = {
    "100": "BaselineInic", "110": "Jitter", "120": "Video", "130": "Decisao",
    "140": "RespA", "141": "RespB", "142": "RespC", "143": "RespD",
    "150": "FixPost", "160": "BaselineFin"
}

dados_mestre_fnirs = []

print(f"A iniciar processamento neurofisiológico (fNIRS) em lote para {NUM_SUJEITOS} sujeitos...\n")

# ==========================================
# 2. Ciclo de Processamento
# ==========================================
for i in range(1, NUM_SUJEITOS + 1):
    sujeito_id = f"sub-{i:03d}"
    pasta_nirs = os.path.join(BASE_DIR, sujeito_id, "nirs")
    
    print(f"[{sujeito_id}] A processar fNIRS...")
    
    try:
        # -- 2.1 Identificação Dinâmica da Pasta --
        if not os.path.exists(pasta_nirs):
            raise FileNotFoundError("Pasta 'nirs' não existe.")
            
        pastas_dentro_nirs = os.listdir(pasta_nirs)
        nome_pasta_run = next((p for p in pastas_dentro_nirs if TAREFA in p and os.path.isdir(os.path.join(pasta_nirs, p))), None)
        
        if not nome_pasta_run:
            raise FileNotFoundError(f"Pasta da tarefa '{TAREFA}' não encontrada.")
            
        pasta_alvo = os.path.join(pasta_nirs, nome_pasta_run)
        ficheiros_pasta = os.listdir(pasta_alvo)
        snirf_file = next((f for f in ficheiros_pasta if f.endswith('.snirf')), None)
        
        if not snirf_file:
            raise FileNotFoundError("Ficheiro .snirf em falta.")
            
        caminho_snirf = os.path.join(pasta_alvo, snirf_file)

        # -- 2.2 Carregamento e Qualidade (SCI) --
        raw_nirs = mne.io.read_raw_snirf(caminho_snirf, preload=True, verbose=False)
        raw_od = optical_density(raw_nirs)
        
        sci = scalp_coupling_index(raw_od)
        raw_od.info['bads'] = list(np.compress(sci < LIMIAR_SCI, raw_od.ch_names))

        # -- 2.3 TDDR e Verificação Dinâmica de SSR --
        raw_od_tddr = temporal_derivative_distribution_repair(raw_od)
        
        picks_validos = mne.pick_types(raw_od_tddr.info, fnirs=True, exclude='bads')
        distancias_od = mne.preprocessing.nirs.source_detector_distances(raw_od_tddr.info, picks=picks_validos)
        
        num_curtos_validos = np.sum(distancias_od <= 0.01)
        
        if num_curtos_validos > 0:
            raw_od_ssr = mne_nirs.signal_enhancement.short_channel_regression(raw_od_tddr)
        else:
            raw_od_ssr = raw_od_tddr.copy()

        # -- 2.4 mBLL e Filtro IIR --
        raw_haemo = beer_lambert_law(raw_od_ssr, ppf=PPF)
        raw_filtrado = raw_haemo.filter(LOW_CUTOFF_HZ, HIGH_CUTOFF_HZ, method="iir", verbose=False)

        # -- 2.5 Isolamento Cortical e Triggers --
        picks_fnirs = mne.pick_types(raw_filtrado.info, fnirs=True, exclude="bads")
        distancias = mne.preprocessing.nirs.source_detector_distances(raw_filtrado.info, picks=picks_fnirs)
        picks_longos = picks_fnirs[distancias > DISTANCIA_MIN_LONGA_M]
        
        raw_cerebral = raw_filtrado.copy().pick(picks_longos)
        
        renomear = {cod: nome for cod, nome in EVENTOS_DM.items() if cod in raw_cerebral.annotations.description}
        raw_cerebral.annotations.rename(renomear)

        # -- 2.6 GLM (Glover) --
        design_matrix = make_first_level_design_matrix(
            raw_cerebral,
            drift_model="cosine",
            high_pass=LOW_CUTOFF_HZ,
            hrf_model="glover"
        )
        
        glm_est = run_glm(raw_cerebral, design_matrix)
        
        colunas = design_matrix.columns.tolist()
        if "Video" not in colunas:
            print(f"  [{sujeito_id}] AVISO: Marcador 'Video' não detetado na matriz. A saltar GLM...")
            continue
            
        vetor_contraste = np.zeros(len(colunas))
        vetor_contraste[colunas.index("Video")] = 1
        contraste = glm_est.compute_contrast(vetor_contraste)
        df_contraste = contraste.to_dataframe()

        # -- 2.7 Extração de Betas e p-values para HbO --
        df_video_hbo = df_contraste[df_contraste['Chroma'] == 'hbo']
        
        registo = {
            'ID': sujeito_id,
            'Tarefa': 'Decision_Making',
            'Canais_Rejeitados_SCI': len(raw_od.info['bads'])
        }
        
        for _, row in df_video_hbo.iterrows():
            canal = f"S{row['Source']}_D{row['Detector']}"
            registo[f'HbO_{canal}_Beta'] = row['effect']
            registo[f'HbO_{canal}_pval'] = row['p_value']
                
        dados_mestre_fnirs.append(registo)
        print(f"  [{sujeito_id}] GLM processado. Canais maus (SCI): {len(raw_od.info['bads'])}.")
        
    except Exception as e:
        print(f"  [{sujeito_id}] ERRO: {e}")

# ==========================================
# 3. Exportação da Base de Dados Consolidada
# ==========================================
if dados_mestre_fnirs:
    df_mestre_fnirs = pd.DataFrame(dados_mestre_fnirs)
    pasta_exportacao = os.path.join(BASE_DIR, "derivatives", "mne-nirs")
    os.makedirs(pasta_exportacao, exist_ok=True)
    caminho_csv_final = os.path.join(pasta_exportacao, "dataset_mestre_fnirs_DM.csv")
    df_mestre_fnirs.to_csv(caminho_csv_final, index=False)
    
    print(f"\nProcessamento fNIRS Concluído! Dados extraídos de {len(dados_mestre_fnirs)} sujeitos.")
    print(f"Base de dados exportada para:\n{caminho_csv_final}")
else:
    print("\nNenhum sujeito obteve dados válidos de fNIRS para exportação.")

A iniciar processamento neurofisiológico (fNIRS) em lote para 54 sujeitos...

[sub-001] A processar fNIRS...
  [sub-001] GLM processado. Canais maus (SCI): 18.
[sub-002] A processar fNIRS...
  [sub-002] GLM processado. Canais maus (SCI): 16.
[sub-003] A processar fNIRS...
  [sub-003] GLM processado. Canais maus (SCI): 16.
[sub-004] A processar fNIRS...
  [sub-004] GLM processado. Canais maus (SCI): 16.
[sub-005] A processar fNIRS...
  [sub-005] GLM processado. Canais maus (SCI): 16.
[sub-006] A processar fNIRS...
  [sub-006] GLM processado. Canais maus (SCI): 16.
[sub-007] A processar fNIRS...
  [sub-007] GLM processado. Canais maus (SCI): 16.
[sub-008] A processar fNIRS...
  [sub-008] GLM processado. Canais maus (SCI): 16.
[sub-009] A processar fNIRS...
  [sub-009] GLM processado. Canais maus (SCI): 16.
[sub-010] A processar fNIRS...
  [sub-010] GLM processado. Canais maus (SCI): 16.
[sub-011] A processar fNIRS...
  [sub-011] GLM processado. Canais maus (SCI): 16.
[sub-012] A processa